In [1]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import re
import joblib

In [2]:
MODEL1_PATH = "/kaggle/input/byt5-base-big-data2"
MODEL2_PATH = "/kaggle/input/byt5-akkadian-model"
MODEL3_PATH = "/kaggle/input/train-gap-all-2/byt5-base-akkadian_gap_setence2"

In [3]:
USE_AMP = False

In [5]:
def replace_gaps(text):
    if pd.isna(text): 
        return text
    
    text = re.sub(r'\.3(?:\s+\.3)+\.{3}(?:\s+\.{3})+\s+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'\.3(?:\s+\.3)+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'\.{3}(?:\s+\.{3})+', '<big_gap>', text)

    text = re.sub(r'xx', '<gap>', text)
    text = re.sub(r' x ', ' <gap> ', text)
    text = re.sub(r'……', '<big_gap>', text)
    text = re.sub(r'\.\.\.\.\.\.', '<big_gap>', text)
    text = re.sub(r'…', '<big_gap>', text)
    text = re.sub(r'\.\.\.', '<big_gap>', text)

    return text

In [6]:
TEST_DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
BATCH_SIZE = 8
MAX_LENGTH = 512
DEVICE = torch.device("cuda")

perf1 = 0.80
perf2 = 1.00
perf3 = 0.55

total = perf1 + perf2 + perf3
w1 = perf1 / total
w2 = perf2 / total
w3 = perf3 / total

print("Loading Model 1...")
model1 = AutoModelForSeq2SeqLM.from_pretrained(MODEL1_PATH)
sd1 = model1.state_dict()

print("Loading Model 2...")
model2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)
sd2 = model2.state_dict()

print("Loading Model 3...")
model3 = AutoModelForSeq2SeqLM.from_pretrained(MODEL3_PATH)
sd3 = model3.state_dict()

print("Averaging weights...")
final_sd = sd2.copy()

for key in final_sd:
    if key in sd1 and key in sd3:
        final_sd[key] = (
            w1 * sd1[key] +
            w2 * sd2[key] +
            w3 * sd3[key]
        )
    elif key in sd1:
        final_sd[key] = w1 * sd1[key] + (w2 + w3) * sd2[key]  # fallback
    elif key in sd3:
        final_sd[key] = w3 * sd3[key] + (w1 + w2) * sd2[key]

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)
model.load_state_dict(final_sd)
model.to(DEVICE).eval()
model.float() 

tokenizer = AutoTokenizer.from_pretrained(MODEL2_PATH)

test_df = pd.read_csv(TEST_DATA_PATH)
test_df['transliteration'] = test_df['transliteration'].apply(replace_gaps)

Loading Model 1...
Loading Model 2...
Loading Model 3...
Averaging weights...


In [7]:
PREFIX = "translate Akkadian to English: "

class InferenceDataset(Dataset):
    def __init__(self, df):
        self.ids = df["id"].tolist()
        self.texts = [PREFIX + t for t in df["transliteration"].astype(str).tolist()]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.ids[idx], self.texts[idx]

def collate_fn(batch):
    ids, texts = zip(*batch)
    enc = tokenizer(
        list(texts),
        max_length=MAX_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    return list(ids), enc["input_ids"], enc["attention_mask"]

loader = DataLoader(
    InferenceDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_fn
)

print("Starting Inference...")

Starting Inference...


In [8]:
all_ids, all_pred = [], []

torch.set_grad_enabled(False)

with torch.inference_mode():
    for ids, input_ids, attention_mask in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        if USE_AMP:
            ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
        else:
            from contextlib import nullcontext
            ctx = nullcontext()

        with ctx:
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                length_penalty=1.05,
                max_new_tokens=384,
                num_beams=6,
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        decoded = [d.strip() if d.strip() else "broken text" for d in decoded]

        all_ids.extend(ids)
        all_pred.extend(decoded)

In [9]:
submission = pd.DataFrame(
    {
        "id": all_ids,
        "translation": all_pred
    }
)

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head())

Saved submission.csv
   id                                        translation
0   0  From the Kanesh colony to the <big_gap> of our...
1   1  In the tablet of the City you wrote to me in t...
2   2  Just as you hear our letter, he has given eith...
3   3  I sent our tablets to every single or two or t...
